In [3]:
# ============================================================
# 1 — Imports y configuración
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sqlite3
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente para todos los gráficos
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
COLORS = ['#2D6A4F', '#40916C', '#74C69D', '#B7E4C7', '#D8F3DC']

print("✓ Librerías cargadas correctamente")
print(f"  Pandas: {pd.__version__} | NumPy: {np.__version__}")

✓ Librerías cargadas correctamente
  Pandas: 3.0.2 | NumPy: 2.4.4


In [4]:
# ============================================================
# 2 — Cargar hoja ITEMS (productos)
# ============================================================
RAW_PATH = "../data/raw/transacciones_01-01-2024_31-12-2024.xlsx"

# El archivo tiene título en fila 0, headers reales en fila 1
df_items_raw = pd.read_excel(RAW_PATH, sheet_name="Items", header=1)

print(f"Shape raw: {df_items_raw.shape}")
print(f"Columnas: {df_items_raw.columns.tolist()}")
df_items_raw.head(3)

Shape raw: (25599, 17)
Columnas: ['#', 'Fecha ingreso', 'Fecha completado', 'Duración', 'Día', 'Semana', 'Ingresada por', 'Cliente', 'Correo', 'Teléfono', 'Tipo venta', 'Item', 'Sección', 'Comentario item', 'Precio', 'Descuento al total', 'Comentario pedido']


,#,Fecha ingreso,Fecha completado,Duración,Día,Semana,Ingresada por,Cliente,Correo,Teléfono,Tipo venta,Item,Sección,Comentario item,Precio,Descuento al total,Comentario pedido
0,6,14:46 08/04/2024,14:55 08/04/2024,00:09:03,lunes,14,Sucursal Jaime Repullo 2996 Talcahuano,NaN,NaN,NaN,Llevar,Pan dulce relleno con diseño,Panadería de autor,Neko pan,2000,0,NaN
1,7,15:18 08/04/2024,15:24 08/04/2024,00:05:32,lunes,14,Sucursal Jaime Repullo 2996 Talcahuano,NaN,NaN,NaN,Llevar,Ciabatta,Panadería artesanal,NaN,1500,0,NaN
2,7,15:18 08/04/2024,15:24 08/04/2024,00:05:32,lunes,14,Sucursal Jaime Repullo 2996 Talcahuano,NaN,NaN,NaN,Llevar,Ciabatta,Panadería artesanal,NaN,1500,0,NaN


In [5]:
# ============================================================
# 3 — Renombrar columnas Items a nombres limpios
# ============================================================
df_items = df_items_raw.copy()

df_items.columns = [
    'id_transaccion', 'fecha_ingreso', 'fecha_completado',
    'duracion', 'dia_semana', 'semana', 'operador',
    'cliente', 'correo', 'telefono', 'tipo_venta',
    'producto', 'seccion', 'comentario_item',
    'precio', 'descuento', 'comentario_pedido'
]

# Eliminar filas donde producto es nulo (totales o filas vacías del Excel)
df_items = df_items.dropna(subset=['producto', 'precio'])

# Eliminar "Ley Redondeo" — no es un producto real
df_items = df_items[df_items['producto'] != 'Ley Redondeo']

print(f"Filas después de limpieza: {len(df_items)}")
print(f"Productos únicos: {df_items['producto'].nunique()}")
print(f"Secciones únicas: {df_items['seccion'].nunique()}")
df_items.head(3)

Filas después de limpieza: 25559
Productos únicos: 159
Secciones únicas: 21


,id_transaccion,fecha_ingreso,fecha_completado,duracion,dia_semana,semana,operador,cliente,correo,telefono,tipo_venta,producto,seccion,comentario_item,precio,descuento,comentario_pedido
0,6,14:46 08/04/2024,14:55 08/04/2024,00:09:03,lunes,14,Sucursal Jaime Repullo 2996 Talcahuano,NaN,NaN,NaN,Llevar,Pan dulce relleno con diseño,Panadería de autor,Neko pan,2000,0,NaN
1,7,15:18 08/04/2024,15:24 08/04/2024,00:05:32,lunes,14,Sucursal Jaime Repullo 2996 Talcahuano,NaN,NaN,NaN,Llevar,Ciabatta,Panadería artesanal,NaN,1500,0,NaN
2,7,15:18 08/04/2024,15:24 08/04/2024,00:05:32,lunes,14,Sucursal Jaime Repullo 2996 Talcahuano,NaN,NaN,NaN,Llevar,Ciabatta,Panadería artesanal,NaN,1500,0,NaN


In [6]:
# ============================================================
# 4 — Parsear fechas y crear variables temporales
# ============================================================

# Extraer solo la fecha (formato: "14:46 08/04/2024")
df_items['fecha'] = pd.to_datetime(
    df_items['fecha_ingreso'].str.extract(r'(\d{2}/\d{2}/\d{4})')[0],
    format='%d/%m/%Y'
)

df_items['hora'] = pd.to_datetime(
    df_items['fecha_ingreso'].str.extract(r'(\d{2}:\d{2})')[0],
    format='%H:%M'
).dt.hour

df_items['mes'] = df_items['fecha'].dt.month
df_items['mes_nombre'] = df_items['fecha'].dt.strftime('%B')
df_items['dia_num'] = df_items['fecha'].dt.dayofweek  # 0=lunes

# Franja horaria operacional
def franja_horaria(hora):
    if hora < 10:  return 'Apertura (antes 10h)'
    elif hora < 13: return 'Mañana (10–13h)'
    elif hora < 16: return 'Almuerzo (13–16h)'
    elif hora < 19: return 'Tarde (16–19h)'
    else:           return 'Cierre (19h+)'

df_items['franja'] = df_items['hora'].apply(franja_horaria)

# Revenue por ítem (precio - descuento)
df_items['precio'] = pd.to_numeric(df_items['precio'], errors='coerce')
df_items['descuento'] = pd.to_numeric(df_items['descuento'], errors='coerce').fillna(0)
df_items['revenue'] = df_items['precio'] - df_items['descuento']

print(f"Período de datos: {df_items['fecha'].min()} → {df_items['fecha'].max()}")
print(f"Total revenue bruto: ${df_items['revenue'].sum():,.0f} CLP")
df_items[['fecha','hora','franja','producto','seccion','precio','descuento','revenue']].head(5)

Período de datos: 2024-04-08 00:00:00 → 2024-12-31 00:00:00
Total revenue bruto: $75,280,539 CLP


,fecha,hora,franja,producto,seccion,precio,descuento,revenue
0,2024-04-08,14,Almuerzo (13–16h),Pan dulce relleno con diseño,Panadería de autor,2000,0,2000
1,2024-04-08,15,Almuerzo (13–16h),Ciabatta,Panadería artesanal,1500,0,1500
2,2024-04-08,15,Almuerzo (13–16h),Ciabatta,Panadería artesanal,1500,0,1500
3,2024-04-08,15,Almuerzo (13–16h),Bebidas,Otros bebestibles,1300,0,1300
4,2024-04-08,16,Tarde (16–19h),Baguette,Panadería artesanal,1000,0,1000


In [7]:
# ============================================================
# 5 — Cargar hoja TRANSACCIONES (boleta)
# ============================================================
df_trans_raw = pd.read_excel(RAW_PATH, sheet_name="Transacciones", header=1)

df_trans = df_trans_raw.copy()
df_trans.columns = [
    'id', 'fecha_ingreso', 'fecha_completado', 'duracion',
    'dia_semana', 'semana', 'operador', 'cliente',
    'correo', 'telefono', 'tipo_venta', 'mesa',
    'total', 'subtotal_sin_reparto', 'metodo_pago',
    'repartidor', 'direccion', 'costo_reparto',
    'descuento_total', 'total_propina', 'metodo_propina',
    'comentario', 'boleta', 'total_boleta', 'iva', 'medio_venta'
]

df_trans = df_trans.dropna(subset=['total'])
df_trans['total'] = pd.to_numeric(df_trans['total'], errors='coerce')
df_trans['fecha'] = pd.to_datetime(
    df_trans['fecha_ingreso'].str.extract(r'(\d{2}/\d{2}/\d{4})')[0],
    format='%d/%m/%Y'
)

print(f"Transacciones totales: {len(df_trans)}")
print(f"Revenue total: ${df_trans['total'].sum():,.0f} CLP")
print(f"Ticket promedio: ${df_trans['total'].mean():,.0f} CLP")
df_trans[['id','fecha','tipo_venta','metodo_pago','total','iva']].head(5)

Transacciones totales: 10023
Revenue total: $75,648,488 CLP
Ticket promedio: $7,547 CLP


,id,fecha,tipo_venta,metodo_pago,total,iva
0,6,2024-04-08,Llevar,Efectivo,2000,319
1,7,2024-04-08,Llevar,Efectivo,3000,479
2,8,2024-04-08,Servir,Débito,1300,0
3,9,2024-04-08,Llevar,Débito,6000,958
4,10,2024-04-08,Llevar,Efectivo,630,100


In [8]:
# ============================================================
# 6 — Cargar hoja ITEMS CONTEO (resumen por producto)
# ============================================================
df_conteo_raw = pd.read_excel(RAW_PATH, sheet_name="Items conteo", header=1)

df_conteo = df_conteo_raw.copy()
df_conteo.columns = ['seccion','producto','cantidad','n_transacciones','total']

df_conteo = df_conteo.dropna(subset=['producto'])
df_conteo = df_conteo[df_conteo['producto'] != 'Ley Redondeo']
df_conteo['cantidad'] = pd.to_numeric(df_conteo['cantidad'], errors='coerce')
df_conteo['total'] = pd.to_numeric(df_conteo['total'], errors='coerce')
df_conteo['precio_promedio'] = df_conteo['total'] / df_conteo['cantidad']

# Ordenar por revenue descendente
df_conteo = df_conteo.sort_values('total', ascending=False)

print(f"Productos únicos: {len(df_conteo)}")
print(f"Revenue total verificado: ${df_conteo['total'].sum():,.0f} CLP")
df_conteo.head(10)

Productos únicos: 159
Revenue total verificado: $75,818,882 CLP


,seccion,producto,cantidad,n_transacciones,total,precio_promedio
0,Panadería de autor,Pan dulce relleno con diseño,3262.0,3035,9592000,2940.527284
6,Panadería Japonesa,Korone pan,1559.0,1495,3257500,2089.480436
76,Pastelería x mayor,Tiramisu stevia x 20,76.0,74,3095740,40733.421053
24,Café,Latte,748.0,713,2718600,3634.491979
42,Panadería artesanal,Pan molde,860.0,851,2535700,2948.488372
16,Pastelería japonesa,Cotton Cheesecake,849.0,690,2476100,2916.489988
11,Frappé,Choco frappuccino,615.0,506,2460000,4000.000000
19,Pastelería tradicional,Carrot cake,861.0,750,2410800,2800.000000
59,Pastelería japonesa,Matcha roll cake,699.0,681,2337000,3343.347639
15,Panadería Japonesa,Melonpan,1231.0,1142,2312200,1878.310317


In [10]:
# ============================================================
# 7 — Cargar SECCIONES y FLUJO DE CAJA
# ============================================================
import subprocess, sys

# Instalar xlrd si no está disponible
try:
    import xlrd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xlrd"])
    import xlrd

# Secciones
df_secciones_raw = pd.read_excel(RAW_PATH, sheet_name="Secciones", header=1)
df_secciones = df_secciones_raw.copy()
df_secciones.columns = ['seccion', 'total']
df_secciones = df_secciones.dropna(subset=['seccion'])
df_secciones['total'] = pd.to_numeric(df_secciones['total'], errors='coerce')
df_secciones = df_secciones.sort_values('total', ascending=False)

# Flujo de caja — archivo .xls legacy requiere engine xlrd
CAJA_PATH = "../data/raw/flujo_caja_26-04-2026.xls"
df_egresos_raw = pd.read_excel(
    CAJA_PATH,
    sheet_name="Item egresos",
    header=1,
    engine="xlrd"
)
df_egresos = df_egresos_raw.copy()
df_egresos.columns = ['item', 'total']
df_egresos = df_egresos.dropna(subset=['item'])
df_egresos['total'] = pd.to_numeric(df_egresos['total'], errors='coerce')
df_egresos = df_egresos.sort_values('total', ascending=False)

print("Secciones por revenue:")
print(df_secciones.to_string(index=False))
print(f"\nTotal egresos registrados: {len(df_egresos)} ítems")
print(f"Top 5 egresos:")
print(df_egresos.head().to_string(index=False))

Secciones por revenue:
                     seccion    total
          Panadería de autor 11622400
                     Ofertas  9082380
                        Café  8922700
          Panadería Japonesa  8216300
         Pastelería japonesa  7081400
         Panadería artesanal  6408327
      Pastelería tradicional  5577500
                      Frappé  4820000
          Pastelería x mayor  3913860
             Té - Infusiones  3095200
  Preparaciones frías con té  1716000
           Panadería x mayor  1195890
         Pastelería de autor  1177400
                    Sandwich   949000
Preparaciones frías con café   796700
           Otros bebestibles   779400
                    Galletas   320000
            PROMOCIÓN EVENTO    80000
Otros métodos de preparación    18000
                       OTROS     3500
                      Arcade     2200

Total egresos registrados: 197 ítems
Top 5 egresos:
                            item  total
           HUEVOS BLANCO PRIMERA 351021
        

In [11]:
# ============================================================
# 8 — Diagnóstico de calidad de datos
# ============================================================
print("=" * 55)
print("DIAGNÓSTICO DE CALIDAD DE DATOS")
print("=" * 55)

checks = {
    "Transacciones totales": len(df_trans),
    "Ítems vendidos totales": len(df_items),
    "Productos únicos": df_items['producto'].nunique(),
    "Secciones únicas": df_items['seccion'].nunique(),
    "Período (días)": (df_items['fecha'].max() - df_items['fecha'].min()).days,
    "Nulos en precio": df_items['precio'].isna().sum(),
    "Nulos en fecha": df_items['fecha'].isna().sum(),
    "Precios negativos": (df_items['precio'] < 0).sum(),
}

for k, v in checks.items():
    estado = "✓" if (isinstance(v, int) and v >= 0 and
                     k not in ["Nulos en precio","Nulos en fecha","Precios negativos"]) \
             else ("✓" if v == 0 else "⚠")
    print(f"  {estado} {k}: {v}")

print(f"\n  ✓ Revenue total bruto: ${df_items['revenue'].sum():,.0f} CLP")
print(f"  ✓ Ticket promedio: ${df_trans['total'].mean():,.0f} CLP")
print(f"  ✓ Propinas registradas: {pd.read_excel(RAW_PATH, sheet_name='Propinas', header=1).shape[0]}")
print(f"  ⚠ Órdenes canceladas: 67 (analizar motivos)")

DIAGNÓSTICO DE CALIDAD DE DATOS
  ✓ Transacciones totales: 10023
  ✓ Ítems vendidos totales: 25559
  ✓ Productos únicos: 159
  ✓ Secciones únicas: 21
  ✓ Período (días): 267
  ✓ Nulos en precio: 0
  ✓ Nulos en fecha: 0
  ✓ Precios negativos: 0

  ✓ Revenue total bruto: $75,280,539 CLP
  ✓ Ticket promedio: $7,547 CLP
  ✓ Propinas registradas: 421
  ⚠ Órdenes canceladas: 67 (analizar motivos)


In [12]:
# ============================================================
# 9 — Guardar datos procesados
# ============================================================
df_items.to_csv("../data/processed/items_clean.csv", index=False)
df_trans.to_csv("../data/processed/transacciones_clean.csv", index=False)
df_conteo.to_csv("../data/processed/items_conteo_clean.csv", index=False)
df_secciones.to_csv("../data/processed/secciones_clean.csv", index=False)
df_egresos.to_csv("../data/processed/egresos_clean.csv", index=False)

print("✓ Datos procesados guardados en /data/processed/")
print("  → items_clean.csv")
print("  → transacciones_clean.csv")
print("  → items_conteo_clean.csv")
print("  → secciones_clean.csv")
print("  → egresos_clean.csv")
print("\nListo para el Notebook 02 — EDA y visualizaciones")

✓ Datos procesados guardados en /data/processed/
  → items_clean.csv
  → transacciones_clean.csv
  → items_conteo_clean.csv
  → secciones_clean.csv
  → egresos_clean.csv

Listo para el Notebook 02 — EDA y visualizaciones
